# Análisis estadístico – índice de profesionalización y participación

Este notebook documenta el análisis estadístico del proyecto *perfil-profesionalizacion-partidos-ecuador*:

- Cálculo de correlaciones Pearson y Spearman entre la profesionalización media por partido/provincia y la variación de participación electoral.
- Ajuste de una regresión lineal multivariada con controles socioeconómicos del INEC.
- Visualizaciones de diagnóstico (dispersión, residuos, influencia).


In [ ]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import statsmodels.api as sm
from scipy.stats import pearsonr, spearmanr

DATA_PROCESSED = Path("data/processed")
merged_path = DATA_PROCESSED / "agg_prof_turnout_with_inec.parquet"

df = pd.read_parquet(merged_path)
df.head()


In [ ]:
# Correlaciones Pearson y Spearman
x = df["profesionalizacion_media"].values
y = df["delta_participacion"].values

pearson_corr, pearson_p = pearsonr(x, y)
spearman_corr, spearman_p = spearmanr(x, y)

pearson_corr, pearson_p, spearman_corr, spearman_p


In [ ]:
# Regresión lineal multivariada con controles socioeconómicos
y = df["delta_participacion"]
X = df[["profesionalizacion_media", "gdp_per_capita", "poverty_rate"]]
X = sm.add_constant(X)

model = sm.OLS(y, X, missing="drop").fit()
model.summary()


In [ ]:
# Dispersión profesionalización vs delta de participación
fig = px.scatter(
    df,
    x="profesionalizacion_media",
    y="delta_participacion",
    color="province",
    hover_data=["party_normalized"],
    title="Profesionalización media vs delta de participación",
)
fig.show()
